# MCA-YOLO-A Exp1 云端训练

仅替换 MobileNetV3 主干网络；原 `train_colab.ipynb` 仍用于 Baseline。

In [ ]:
# 1. 检查 GPU 并安装依赖
import torch
print(torch.__version__, torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('请在 Colab 中选择 T4 GPU')
!pip install ultralytics -q
from ultralytics import YOLO

In [ ]:
# 2. 拉取项目并配置云端数据路径
import os
REPO_PATH = '/content/mca-yolo-reproduce'
if not os.path.exists(REPO_PATH):
    !git clone https://github.com/Cranzz/mca-yolo-reproduce.git {REPO_PATH}
else:
    !cd {REPO_PATH} && git pull
%cd {REPO_PATH}
yaml_path = f'{REPO_PATH}/data/rdd2022.yaml'
with open(yaml_path, 'w') as f:
    f.write(f'''path: {REPO_PATH}/data/yolo_format\ntrain: train/images\nval: val/images\ntest: test/images\nnc: 4\nnames: ['D00', 'D10', 'D20', 'D40']''')

In [ ]:
# 3. 训练 Exp1：YOLOv8n + MobileNetV3
model = YOLO(f'{REPO_PATH}/models/yolov8n_mobilenetv3.yaml')
results = model.train(data=yaml_path, epochs=100, imgsz=640, batch=16, name='yolov8n_rdd2022_exp1_mobilenetv3_colab', device=0, plots=True)
print(f'最佳模型: {results.save_dir}')

In [ ]:
# 4. 测试集评估
metrics = model.val(data=yaml_path, split='test', imgsz=640, batch=16, device=0)
print(f'mAP50: {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')